<h2>Makes an overview of all the reconstructions and data, then saves it to a file.</h2>

In [1]:
import os
import numpy as np
import re
from ptypy import io
import glob

WARNING ptypy - Message Passaging for Python (mpi4py) not found.
    CPU-parallelization disabled.
    Install python-mpi4py via the package repositories or with `pip install --user mpi4py`


In [3]:
def sort_condition(string):
    string_cond = string.rsplit('_',1)[-1]
    return string_cond


def get_rec_data(paths):
    allsubpaths = []
    allpath_recs = []
    nriter = []
    recfolders = []
    ####expdata = {}
    expdata = []
    for path in paths:
        # loop through all 65 live reconstructions
        subpaths = glob.glob(path + '*/')
        subpaths.sort()
        allsubpaths.append(subpaths)
        # print(str(path.rsplit('/',2)[1]).ljust(20,' '), len(subpaths), subpaths,'\n')
        for subpath in subpaths:
            try:
                #path_rec = glob.glob(subpath + f'rec/*')[0] ## assunimg there is only one rec file here
                path_rec = glob.glob(subpath + f'rec/*')
                path_rec.sort(key=sort_condition)  # now returns a list of every engines final reconstruction path, sorted with increasing number of iterations.
                allpath_recs.append(path_rec)

                nriter.append(int(path_rec[-1].rsplit('_', 1)[-1].rsplit('.ptyr', 1)[0]))

                recfolder = path_rec[0].rsplit('/',3)[1]
                recfolders.append(recfolder) # is of the form 'XXXXXX_XX'
                scan = recfolder.split('_')[0]
                rawpaths = glob.glob('/data/visitors/nanomax/20250057/2025021508/raw/' + f'*/*{scan}*')
                rawpaths.sort()
                sample = path_rec[0].rsplit('/', 4)[1]

                obj = 0###io.h5read(path_rec, 'content/obj/Sscan00G00/data')['content/obj/Sscan00G00/data']
                #probe = obj = io.h5read(path_rec, 'content/probe/Sscan00G00/data')['content/probe/Sscan00G00/data']
                ####expdata[recfolder] = {'sample': sample, 'nriter': nriter[-1], 'obj': obj}
                expdata.append({'scan': scan, 'recfoldername': recfolder, 'sample': sample, 'nriter': nriter[-1], 'rec_path': path_rec, 'rawdata_paths': rawpaths})
            except:
                print(f'No rec file in {subpath}')
    return expdata, recfolders, nriter, allpath_recs, allsubpaths

samps = ['siemens', 'siemens_big', '10micron_tungsten', '5micron_tungsten', 'fossils', 'broken_gold', 'siemens_incoherent'] # to get a data retrieved in the same order as the scans
base_live = "/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/"
base_livesim = "/data/visitors/nanomax/20250057/2025021508/process/RL/LiveSimPtycho/"
base_offl = "/data/visitors/nanomax/20250057/2025021508/process/RL/OfflinePtycho/"

paths_live = [base_live+f'{samp}/' for samp in samps]  # glob.glob(base_live+'*/')
paths_livesim = [base_livesim+f'{samp}/' for samp in samps]  # glob.glob(base_live+'*/')
paths_offl = [base_offl+f'{samp}/' for samp in samps]  # glob.glob(base_offl+'*/')


expdata_live, recfolders_live, nriter_live, allpath_recs_live, allsubpaths_live = get_rec_data(paths_live)
expdata_livesim, recfolders_livesim, nriter_livesim, allpath_recs_livesim, allsubpaths_livesim = get_rec_data(paths_livesim)
expdata_offl, recfolders_offl, nriter_offl, allpath_recs_offl, allsubpaths_offl = get_rec_data(paths_offl)

# Save to file:

np.savez("recon_info.npz", live_info=expdata_live, livesim_info=expdata_livesim, offl_info=expdata_offl)

#print('live', allpath_recs_live)
#print(f'maximum nr of iterations _live: {np.max(nriter_live)}')
#print('\noffline', allpath_recs_offl)
#print(f'maximum nr of iterations _offl: {np.max(nriter_offl)}')
#



No rec file in /data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/000013_00_stopped/
No rec file in /data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/000014_00_stopped/
No rec file in /data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/000017_00/
No rec file in /data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/000019_00_stopped/
No rec file in /data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/000020_00_stopped/
No rec file in /data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/000021_00_stopped/
No rec file in /data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/000022_00_stopped/
No rec file in /data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/000023_00_stopped/


In [16]:
path_rec[0].rsplit('_',1)[-1]

'1221.ptyr'

<h2>Example usages</h2>

In [8]:
# Load data from file
npzfile = np.load("recon_info.npz", allow_pickle=True)
print(npzfile.files)  # prints the keys

live_info = list(npzfile['live_info'])
livesim_info = list(npzfile['livesim_info'])
offl_info = list(npzfile['offl_info'])

['live_info', 'livesim_info', 'offl_info']


In [20]:
# Extract subset based on scannumber:
#scanset = [recon for recon in offl_info+live_info if recon["sample"]=="siemens_big"]
reclist = [recon for recon in live_info+livesim_info+offl_info if recon["scan"]=="000027"]


# Extract subset based on sample type
#scanset = [recon for recon in live_info+livesim_info+offl_info if recon["sample"]=="siemens_big"]
reclist_live = [recon for recon in live_info if recon["sample"]=="siemens"]
reclist_livesim = [recon for recon in livesim_info if recon["sample"]=="siemens"]
reclist_offl = [recon for recon in offl_info if recon["sample"]=="siemens"]

reclist_offl[0]['rec_path'], reclist_live[0]


# Extract reconstructions  from a subset where 'nriter' > 2000
high_nriter_reconstructions = [recon_dict for recon_dict in reclist_live if recon_dict["nriter"] > 2000]
# Print results
high_nriter_reconstructions

#Extract values "rawdata_paths" from the subset
rawdata_paths = [recon_dict["rawdata_paths"] for recon_dict in reclist_livesim if recon_dict["recfoldername"][-2:] == "01"]

[]

In [36]:
#[recon for recon_list in grouped_data.values() for recon in recon_list if recon["nriter"] > 1000]

high_nriter_reconstructions=[]
for recon_dict in reclist_live:
    #for recon in recon_dict:
    if recon_dict["nriter"] > 2000:
        high_nriter_reconstructions.append(recon_dict)

high_nriter_reconstructions_ = [recon_dict for recon_dict in reclist_live if recon_dict["nriter"] > 2000]
high_nriter_reconstructions_

[{'scan': '000016',
  'recfoldername': '000016_00',
  'sample': 'siemens',
  'nriter': 3400,
  'rec_path': '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/000016_00/rec/rec_scan_000000_DM_pycuda_3400.ptyr',
  'rawdata_paths': ['/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/000016.h5',
   '/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/scan_000016_eiger1m.hdf5']},
 {'scan': '000026',
  'recfoldername': '000026_00',
  'sample': 'siemens',
  'nriter': 2390,
  'rec_path': '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/000026_00/rec/rec_scan_000000_DM_pycuda_2390.ptyr',
  'rawdata_paths': ['/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/scan_000026_eiger1m.hdf5',
   '/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/000026.h5']},
 {'scan': 'XXXXXX',
  'recfoldername': 'XXXXXX_02',
  'sample': 'siemens',
  'nriter': 2490,
  'rec_path': '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtyc

In [34]:
len(reclist_live), reclist_live

(8,
 [{'scan': '000012',
   'recfoldername': '000012_00',
   'sample': 'siemens',
   'nriter': 1678,
   'rec_path': '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/000012_00/rec/rec_scan_000000_DM_pycuda_1678.ptyr',
   'rawdata_paths': ['/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/000012.h5',
    '/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/scan_000012_eiger1m.hdf5']},
  {'scan': '000015',
   'recfoldername': '000015_00',
   'sample': 'siemens',
   'nriter': 1748,
   'rec_path': '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/000015_00/rec/rec_scan_000000_DM_pycuda_1748.ptyr',
   'rawdata_paths': ['/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/scan_000015_eiger1m.hdf5',
    '/data/visitors/nanomax/20250057/2025021508/raw/0001_setup/000015.h5']},
  {'scan': '000016',
   'recfoldername': '000016_00',
   'sample': 'siemens',
   'nriter': 3400,
   'rec_path': '/data/visitors/nanomax/20250057/2025021

<h2>Temporary stuff</h2>

In [ ]:
scanset = []
expdata_offl_0 = expdata_offl[0]
scanset = [recon for recon in expdata_live+expdata_livesim if recon["scan"]=="000055"]
len(scanset), scanset

In [ ]:
expdata_livesim

In [5]:
live, offl = np.load("recon_info.npz", allow_pickle=True)


In [7]:
scanset = []
expdata_offl_0 = expdata_offl[0]
scanset = [recon for recon in expdata_live+expdata_offl if recon["scan"]=="000073"]
len(scanset)
#for dct in expdata_offl_:
#    scanset.append(dct) if dct["scan"] == "000072"
#    dct.
##expdata_offl_
#scanset
#
#high_nriter_reconstructions = [
#    recon for recon_list in grouped_data.values() for recon in recon_list if recon["nriter"] > 1000
#]

4

In [8]:
expdata_offl_0 = expdata_offl_[0]
import collections
expdata_offl_0

SyntaxError: invalid syntax (1287787755.py, line 3)

In [15]:
expdata_offl

from collections import defaultdict


# Restructure data
grouped_data = defaultdict(list)
for key, value in expdata_offl.items():
    grouped_data[value["scan"]].append({"id": key, **value})

# Convert defaultdict to regular dict for cleaner output
grouped_data = dict(grouped_data)

# Print example
grouped_data["broken_gold"]


## Extract reconstructions where 'nriter' > 1000
#high_nriter_reconstructions = [
#    recon for recon_list in grouped_data.values() for recon in recon_list if recon["nriter"] > 1000
#]
#
## Print results
#high_nriter_reconstructions

AttributeError: 'list' object has no attribute 'items'

In [10]:
allpath_recs_offl.sorted()
sorted( employees, key = lambda x : x['Name'] )

AttributeError: 'list' object has no attribute 'sorted'

In [112]:
paths_live

['/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens_incoherent/',
 '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens_big/',
 '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/10micron_tungsten/',
 '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/broken_gold/',
 '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/fossils/',
 '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/5micron_tungsten/',
 '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens/']

In [108]:
allpath_recs_offl[0].removeprefix(base_offl).split('/',1)[-1]

def get_endpart(fname):
    fname.removeprefix(base_offl).split('/',1)[-1]
    
allpath_recs_offl.sort(key=get_endpart())

TypeError: get_endpart() missing 1 required positional argument: 'fname'

In [80]:
print(len(expdata_offl_))
#grouped_data
print(len(allpath_recs_offl))
!ls '/data/visitors/nanomax/20250057/2025021508/process/RL/OfflinePtycho/siemens_incoherent/000073_01/rec'

57
57
rec_scan_000001_DM_cupy_0801.ptyr


In [17]:
expdata_live
expdata_live_list = [
 ['000075_00', 'sample', 'siemens_incoherent', 'nriter', 987, 'obj', 0],
 ['000074_00', 'sample', 'siemens_incoherent', 'nriter', 809, 'obj', 0],
 ['000077_00', 'sample', 'siemens_incoherent', 'nriter', 2000, 'obj', 0],
 ['000076_00', 'sample', 'siemens_incoherent', 'nriter', 752, 'obj', 0],
 ['000078_00', 'sample', 'siemens_incoherent', 'nriter', 2000, 'obj', 0],
 ['000073_00', 'sample', 'siemens_incoherent', 'nriter', 801, 'obj', 0],
 ['000028_00', 'sample', 'siemens_big', 'nriter', 814, 'obj', 0],
 ['000029_00', 'sample', 'siemens_big', 'nriter', 2410, 'obj', 0],
 ['000027_00', 'sample', 'siemens_big', 'nriter', 974, 'obj', 0],
 ['000030_00', 'sample', '10micron_tungsten', 'nriter', 982, 'obj', 0],
 ['000031_00', 'sample', '10micron_tungsten', 'nriter', 1984, 'obj', 0],
 ['000032_00', 'sample', '10micron_tungsten', 'nriter', 1987, 'obj', 0],
 ['000033_00', 'sample', '10micron_tungsten', 'nriter', 1990, 'obj', 0],
 ['000070_00', 'sample', 'broken_gold', 'nriter', 808, 'obj', 0],
 ['000050_00', 'sample', 'broken_gold', 'nriter', 2688, 'obj', 0],
 ['000055_00', 'sample', 'broken_gold', 'nriter', 2477, 'obj', 0],
 ['000072_00', 'sample', 'broken_gold', 'nriter', 808, 'obj', ]
]
expdata_live_list.index()

[['000075_00', 'sample', 'siemens_incoherent', 'nriter', 987, 'obj', 0],
 ['000074_00', 'sample', 'siemens_incoherent', 'nriter', 809, 'obj', 0],
 ['000077_00', 'sample', 'siemens_incoherent', 'nriter', 2000, 'obj', 0],
 ['000076_00', 'sample', 'siemens_incoherent', 'nriter', 752, 'obj', 0],
 ['000078_00', 'sample', 'siemens_incoherent', 'nriter', 2000, 'obj', 0],
 ['000073_00', 'sample', 'siemens_incoherent', 'nriter', 801, 'obj', 0],
 ['000028_00', 'sample', 'siemens_big', 'nriter', 814, 'obj', 0],
 ['000029_00', 'sample', 'siemens_big', 'nriter', 2410, 'obj', 0],
 ['000027_00', 'sample', 'siemens_big', 'nriter', 974, 'obj', 0],
 ['000030_00', 'sample', '10micron_tungsten', 'nriter', 982, 'obj', 0],
 ['000031_00', 'sample', '10micron_tungsten', 'nriter', 1984, 'obj', 0],
 ['000032_00', 'sample', '10micron_tungsten', 'nriter', 1987, 'obj', 0],
 ['000033_00', 'sample', '10micron_tungsten', 'nriter', 1990, 'obj', 0],
 ['000070_00', 'sample', 'broken_gold', 'nriter', 808, 'obj', 0],
 ['0

In [24]:
allpath_recs_live

['/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens_incoherent/000075_00/rec/rec_scan_000000_DM_pycuda_0987.ptyr',
 '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens_incoherent/000074_00/rec/rec_scan_000000_DM_pycuda_0809.ptyr',
 '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens_incoherent/000077_00/rec/rec_scan_000001_DM_pycuda_2000.ptyr',
 '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens_incoherent/000076_00/rec/rec_scan_000000_DM_pycuda_0752.ptyr',
 '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens_incoherent/000078_00/rec/rec_scan_000000_DM_pycuda_2000.ptyr',
 '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens_incoherent/000073_00/rec/rec_scan_000000_DM_pycuda_0801.ptyr',
 '/data/visitors/nanomax/20250057/2025021508/process/RL/LivePtycho/siemens_big/000028_00/rec/rec_scan_000028_DM_pycuda_0814.ptyr',
 '/data/visitors/nanomax/20250057/2025021

<h3>Extract set of reconstructions based on a key:</h3>

In [77]:
from collections import defaultdict


# Restructure data
grouped_data = defaultdict(list)
for key, value in expdata_offl.items():
    grouped_data[value["sample"]].append({"id": key, **value})

# Convert defaultdict to regular dict for cleaner output
grouped_data = dict(grouped_data)

# Print example
len(grouped_data["broken_gold"])



28

In [58]:
# Extract reconstructions where 'nriter' > 1000
high_nriter_reconstructions = [
    recon for recon_list in grouped_data.values() for recon in recon_list if recon["nriter"] > 1000
]

# Print results
high_nriter_reconstructions

[{'id': '000077_00', 'sample': 'siemens_incoherent', 'nriter': 2000, 'obj': 0},
 {'id': '000029_00', 'sample': 'siemens_big', 'nriter': 2410, 'obj': 0},
 {'id': '000031_00', 'sample': '10micron_tungsten', 'nriter': 1984, 'obj': 0},
 {'id': '000032_00', 'sample': '10micron_tungsten', 'nriter': 1987, 'obj': 0},
 {'id': '000033_00', 'sample': '10micron_tungsten', 'nriter': 1990, 'obj': 0},
 {'id': '000050_00', 'sample': 'broken_gold', 'nriter': 2688, 'obj': 0},
 {'id': '000055_00', 'sample': 'broken_gold', 'nriter': 2477, 'obj': 0},
 {'id': '000049_00', 'sample': 'broken_gold', 'nriter': 2510, 'obj': 0},
 {'id': '000045_00', 'sample': 'broken_gold', 'nriter': 2403, 'obj': 0},
 {'id': '000048_00', 'sample': 'broken_gold', 'nriter': 2693, 'obj': 0},
 {'id': '000056_00', 'sample': 'broken_gold', 'nriter': 2710, 'obj': 0},
 {'id': '000047_00', 'sample': 'broken_gold', 'nriter': 2471, 'obj': 0},
 {'id': '000051_00', 'sample': 'broken_gold', 'nriter': 2450, 'obj': 0},
 {'id': '000057_00', 'samp

In [85]:
grouped_data.keys()

dict_keys(['siemens_incoherent', 'siemens_big', '10micron_tungsten', 'broken_gold', 'fossils', '5micron_tungsten', 'siemens'])

In [33]:
expdata_livetst = {
    '000075_00': {'sample': 'siemens_incoherent', 'nriter': 987, 'obj': 0},
    '000074_00': {'sample': 'siemens_incoherent', 'nriter': 809, 'obj': 0},
    '000077_00': {'sample': 'siemens_incoherent', 'nriter': 2000, 'obj': 0},
    '000076_00': {'sample': 'siemens_incoherent', 'nriter': 752, 'obj': 0},
    '000078_00': {'sample': 'siemens_incoherent', 'nriter': 2000, 'obj': 0},
    '000073_00': {'sample': 'siemens_incoherent', 'nriter': 801, 'obj': 0},
    '000028_00': {'sample': 'siemens_big', 'nriter': 814, 'obj': 0},
    '000029_00': {'sample': 'siemens_big', 'nriter': 2410, 'obj': 0},
    '000027_00': {'sample': 'siemens_big', 'nriter': 974, 'obj': 0},
    '000030_00': {'sample': '10micron_tungsten', 'nriter': 982, 'obj': 0},
    '000031_00': {'sample': '10micron_tungsten', 'nriter': 1984, 'obj': 0},
    '000032_00': {'sample': '10micron_tungsten', 'nriter': 1987, 'obj': 0},
    '000033_00': {'sample': '10micron_tungsten', 'nriter': 1990, 'obj': 0},
    '000070_00': {'sample': 'broken_gold', 'nriter': 808, 'obj': 0},
    '000050_00': {'sample': 'broken_gold', 'nriter': 2688, 'obj': 0},
    '000055_00': {'sample': 'broken_gold', 'nriter': 2477, 'obj': 0},
}

# Restructure data
grouped_data = defaultdict(list)
grouped_data



defaultdict(list, {})

In [ ]:
for key, value in expdata_live.items():
    grouped_data[value["sample"]].append({"id": key, **value})
grouped_data

In [ ]:
# Convert defaultdict to regular dict for cleaner output
grouped_data = dict(grouped_data)
grouped_data

In [38]:
# Print example
grouped_data["siemens"]

[{'id': '000015_00', 'sample': 'siemens', 'nriter': 1748, 'obj': 0},
 {'id': 'XXXXXX_02', 'sample': 'siemens', 'nriter': 2490, 'obj': 0},
 {'id': '000026_00', 'sample': 'siemens', 'nriter': 2390, 'obj': 0},
 {'id': '000012_00', 'sample': 'siemens', 'nriter': 1678, 'obj': 0},
 {'id': '000025_00', 'sample': 'siemens', 'nriter': 1731, 'obj': 0},
 {'id': '000024_00', 'sample': 'siemens', 'nriter': 1690, 'obj': 0},
 {'id': '000016_00', 'sample': 'siemens', 'nriter': 3400, 'obj': 0},
 {'id': '000018_00', 'sample': 'siemens', 'nriter': 1703, 'obj': 0},
 {'id': '000015_00', 'sample': 'siemens', 'nriter': 1748, 'obj': 0},
 {'id': 'XXXXXX_02', 'sample': 'siemens', 'nriter': 2490, 'obj': 0},
 {'id': '000026_00', 'sample': 'siemens', 'nriter': 2390, 'obj': 0},
 {'id': '000012_00', 'sample': 'siemens', 'nriter': 1678, 'obj': 0},
 {'id': '000025_00', 'sample': 'siemens', 'nriter': 1731, 'obj': 0},
 {'id': '000024_00', 'sample': 'siemens', 'nriter': 1690, 'obj': 0},
 {'id': '000016_00', 'sample': 'si

In [39]:
grouped_data.keys()

dict_keys(['siemens_incoherent', 'siemens_big', '10micron_tungsten', 'broken_gold', 'fossils', '5micron_tungsten', 'siemens'])